In [0]:
# Trabajo elaborado por Jorge Jaramillo - Johan Orrego
# Verificación de Entorno
import sys
print("version de pyton", sys.version)
print("version de spark", spark.version)


version de pyton 3.12.3 (main, Mar 23 2026, 19:04:32) [GCC 13.3.0]
version de spark 4.1.0


In [0]:
# Definir el esquema
from pyspark.sql.types import *
esquema_natalidad = StructType([
    StructField("Estado", StringType(), False),
    StructField("Total", IntegerType(), False), 
    StructField("Hombres", IntegerType(), False), 
    StructField("Mujeres", IntegerType(), False), 
    StructField("No_especificado", IntegerType(), True), 
    StructField("Year", IntegerType(), False) ])
print ("esquema definido correctamente"),


esquema definido correctamente


(None,)

In [0]:
# Leemos el csv con Spark y modificamos datos erroneos
df = spark.read \
    .option("header", True) \
    .option("inferSchema", False) \
    .csv("/Volumes/workspace/default/data/Natalidad_Mexico.csv")
df = df.withColumnRenamed("No especificado","No_especificado")
df = df.drop("_c6")

from pyspark.sql.functions import col

df = df \
    .withColumn("Total", col("Total").cast("int")) \
    .withColumn("Hombres", col("Hombres").cast("int")) \
    .withColumn("Mujeres", col("Mujeres").cast("int")) \
    .withColumn("No_especificado", col("No_especificado").cast("int")) \
    .withColumn("Year", col("Year").cast("int"))    

# Mostramos el dataframe
print("Total de filas:", df.count())
print("Total de columnas:", len(df.columns))

df.show(10, truncate=False)

Total de filas: 297
Total de columnas: 6
+--------------------+------+-------+-------+---------------+----+
|estado              |Total |Hombres|Mujeres|No_especificado|Year|
+--------------------+------+-------+-------+---------------+----+
|Aguascalientes      |26583 |13603  |12980  |0              |2010|
|Baja California     |63559 |32264  |31295  |0              |2010|
|Baja California Sur |13988 |7076   |6912   |0              |2010|
|Campeche            |20380 |10230  |10150  |0              |2010|
|Coahuila de Zaragoza|56972 |28879  |28093  |0              |2010|
|Colima              |13796 |7105   |6691   |0              |2010|
|Chiapas             |175382|87534  |87789  |59             |2010|
|Chihuahua           |74063 |37236  |36827  |0              |2010|
|Ciudad de México    |160057|79504  |80551  |2              |2010|
|Durango             |42514 |21290  |21224  |0              |2010|
+--------------------+------+-------+-------+---------------+----+
only showing top 10 r

In [0]:
# Creamos una tabla única permanente
df.write.mode("overwrite").saveAsTable("natalidad_flat")

print("La tabla de NATALIDAD_FLAT creada correctamente:", df.count(), "registros")

La tabla de NATALIDAD_FLAT creada correctamente: 297 registros


In [0]:
# Generamos un ID único por registro
from pyspark.sql.functions import monotonically_increasing_id

df_con_id = df.withColumn("nacimiento_id", monotonically_increasing_id())

print("ID generado para", df_con_id.count(), "registros")

df_con_id.show(5, truncate=False)

ID generado para 297 registros
+--------------------+-----+-------+-------+---------------+----+-------------+
|estado              |Total|Hombres|Mujeres|No_especificado|Year|nacimiento_id|
+--------------------+-----+-------+-------+---------------+----+-------------+
|Aguascalientes      |26583|13603  |12980  |0              |2010|0            |
|Baja California     |63559|32264  |31295  |0              |2010|1            |
|Baja California Sur |13988|7076   |6912   |0              |2010|2            |
|Campeche            |20380|10230  |10150  |0              |2010|3            |
|Coahuila de Zaragoza|56972|28879  |28093  |0              |2010|4            |
+--------------------+-----+-------+-------+---------------+----+-------------+
only showing top 5 rows


In [0]:
# creacion de tabka natalidd temporal
df_temporal = df_con_id.select( 
        "nacimiento_id", 
        "Year" ) 
df_temporal.write.mode("overwrite").saveAsTable("natalidad_temporal")
print(
    "Tabla temporal creada correctamente:",
    df_temporal.count(),
    "registros"
)

df_temporal.show(10, truncate=False)

Tabla temporal creada correctamente: 297 registros
+-------------+----+
|nacimiento_id|Year|
+-------------+----+
|0            |2010|
|1            |2010|
|2            |2010|
|3            |2010|
|4            |2010|
|5            |2010|
|6            |2010|
|7            |2010|
|8            |2010|
|9            |2010|
+-------------+----+
only showing top 10 rows


In [0]:
# crecacion de tabla natalidad geografica
df_geografica = df_con_id.select(
     "nacimiento_id", 
     "Estado" ) 
df_geografica.write.mode("overwrite").saveAsTable("natalidad_geografica")
print(
    "Tabla geografica creada correctamente:",
    df_geografica.count(),
    "registros"
)

df_geografica.show(10, truncate=False)


Tabla geografica creada correctamente: 297 registros
+-------------+--------------------+
|nacimiento_id|Estado              |
+-------------+--------------------+
|0            |Aguascalientes      |
|1            |Baja California     |
|2            |Baja California Sur |
|3            |Campeche            |
|4            |Coahuila de Zaragoza|
|5            |Colima              |
|6            |Chiapas             |
|7            |Chihuahua           |
|8            |Ciudad de México    |
|9            |Durango             |
+-------------+--------------------+
only showing top 10 rows


In [0]:
# creacion de tabla natalidad demografica
df_demografica = df_con_id.select( 
        "nacimiento_id",
         "Total", 
         "Hombres", 
         "Mujeres", 
         "No_especificado" )
df_demografica.write.mode("overwrite").saveAsTable("natalidad_demografica")
print(
    "Tabla demografica creada correctamente:",
    df_demografica.count(),
    "registros"
)

df_demografica.show(10, truncate=False)

Tabla demografica creada correctamente: 297 registros
+-------------+------+-------+-------+---------------+
|nacimiento_id|Total |Hombres|Mujeres|No_especificado|
+-------------+------+-------+-------+---------------+
|0            |26583 |13603  |12980  |0              |
|1            |63559 |32264  |31295  |0              |
|2            |13988 |7076   |6912   |0              |
|3            |20380 |10230  |10150  |0              |
|4            |56972 |28879  |28093  |0              |
|5            |13796 |7105   |6691   |0              |
|6            |175382|87534  |87789  |59             |
|7            |74063 |37236  |36827  |0              |
|8            |160057|79504  |80551  |2              |
|9            |42514 |21290  |21224  |0              |
+-------------+------+-------+-------+---------------+
only showing top 10 rows


In [0]:
# Esquema en PySpark
print("===== ESQUEMA TABLA PLANA =====")
df.printSchema()
df_con_id.printSchema()
df_temporal.printSchema()
df_geografica.printSchema()
df_demografica.printSchema()
# Mostramos las tablas creadas
display(spark.sql("SHOW TABLES"))

===== ESQUEMA TABLA PLANA =====
root
 |-- estado: string (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Hombres: integer (nullable = true)
 |-- Mujeres: integer (nullable = true)
 |-- No_especificado: integer (nullable = true)
 |-- Year: integer (nullable = true)

root
 |-- estado: string (nullable = true)
 |-- Total: integer (nullable = true)
 |-- Hombres: integer (nullable = true)
 |-- Mujeres: integer (nullable = true)
 |-- No_especificado: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- nacimiento_id: long (nullable = false)

root
 |-- nacimiento_id: long (nullable = false)
 |-- Year: integer (nullable = true)

root
 |-- nacimiento_id: long (nullable = false)
 |-- Estado: string (nullable = true)

root
 |-- nacimiento_id: long (nullable = false)
 |-- Total: integer (nullable = true)
 |-- Hombres: integer (nullable = true)
 |-- Mujeres: integer (nullable = true)
 |-- No_especificado: integer (nullable = true)



database,tableName,isTemporary
default,delitos_colombia,false
default,delitos_flat,false
default,hurtos_delito,false
default,hurtos_flat,false
default,hurtos_geografica,false
default,hurtos_temporal,false
default,natalidad_demografica,false
default,natalidad_flat,false
default,natalidad_geografica,false
default,natalidad_mexico,false


In [0]:
%sql
-- Esquema en SQL
DESCRIBE TABLE natalidad_flat

col_name,data_type,comment
estado,string,null
Total,int,null
Hombres,int,null
Mujeres,int,null
No_especificado,int,null
Year,int,null
_c6,string,null


In [0]:
%sql
--Consultar los nacimientos por año
SELECT Year, 
SUM(Total) AS Total_Nacimientos
FROM natalidad_flat
 GROUP BY Year ORDER BY Year;

Year,Total_Nacimientos
2010,2643908
2011,2586287
2012,2498880
2013,2478889
2014,2463420
2015,2353596
2016,2293708
2017,2234039
2018,2162535


In [0]:
%sql
-- TOP 10 estados con mas nacimientos
SELECT 
Estado, 
SUM(Total) AS Nacimientos 
FROM natalidad_flat 
GROUP BY Estado 
ORDER BY Nacimientos 
DESC LIMIT 10;

Estado,Nacimientos
México,2780698
Jalisco,1403088
Veracruz de Ignacio de la Llave,1366328
Chiapas,1364795
Puebla,1283782
Ciudad de México,1255560
Guanajuato,1071417
Michoacán de Ocampo,923608
Guerrero,844643
Nuevo León,838343


In [0]:
# TOP 10 estados con mas nacimientos en Pyspark
from pyspark.sql.functions import sum, desc

df_natalidad = spark.table("natalidad_flat")

df_natalidad.groupBy("Estado") \
    .agg(sum("Total").alias("Nacimientos")) \
    .orderBy(desc("Nacimientos")) \
    .limit(10) \
    .show(truncate=False)

+-------------------------------+-----------+
|Estado                         |Nacimientos|
+-------------------------------+-----------+
|México                         |2780698    |
|Jalisco                        |1403088    |
|Veracruz de Ignacio de la Llave|1366328    |
|Chiapas                        |1364795    |
|Puebla                         |1283782    |
|Ciudad de México               |1255560    |
|Guanajuato                     |1071417    |
|Michoacán de Ocampo            |923608     |
|Guerrero                       |844643     |
|Nuevo León                     |838343     |
+-------------------------------+-----------+



In [0]:
%sql
-- consultar estados con mas de 200 mil nacimientos
SELECT Estado, 
Year, 
Total FROM natalidad_flat 
WHERE Total > 200000 
ORDER BY Total DESC;

Estado,Year,Total
México,2010,335898
México,2011,327165
México,2012,326412
México,2013,317834
México,2014,316088
México,2015,303778
México,2016,295635
México,2017,286204
México,2018,271684


In [0]:
%sql
-- consultar los estados donde nacieron mas mujeres que hombres
SELECT Estado, 
Year, 
Hombres, 
Mujeres 
FROM natalidad_flat
WHERE Hombres < Mujeres;

Estado,Year,Hombres,Mujeres
Chiapas,2010,87534,87789
Ciudad de México,2010,79504,80551
México,2010,167792,168105
Michoacán de Ocampo,2010,58090,58170
Morelos,2010,19997,20213
Oaxaca,2010,53592,56032
Puebla,2010,79466,80970
Tlaxcala,2010,13524,13547
Veracruz de Ignacio de la Llave,2010,86708,87375
Campeche,2011,10917,11221


In [0]:
# Total de nacimientos por estado
from pyspark.sql.functions import sum, desc 
df.groupBy("Estado") \
.agg(sum("Total").alias("Nacimientos")) \
.orderBy(desc("Nacimientos")) \
.limit(10) \
.show(truncate=False)

+-------------------------------+-----------+
|Estado                         |Nacimientos|
+-------------------------------+-----------+
|México                         |2780698    |
|Jalisco                        |1403088    |
|Veracruz de Ignacio de la Llave|1366328    |
|Chiapas                        |1364795    |
|Puebla                         |1283782    |
|Ciudad de México               |1255560    |
|Guanajuato                     |1071417    |
|Michoacán de Ocampo            |923608     |
|Guerrero                       |844643     |
|Nuevo León                     |838343     |
+-------------------------------+-----------+



In [0]:
# Promedio de nacimientos por estado
from pyspark.sql.functions import avg, desc

df.groupBy("Estado") \
  .agg(avg("Total").alias("Promedio")) \
  .orderBy(desc("Promedio")) \
  .show(10,truncate=False)

+-------------------------------+------------------+
|Estado                         |Promedio          |
+-------------------------------+------------------+
|México                         |308966.44444444444|
|Jalisco                        |155898.66666666666|
|Veracruz de Ignacio de la Llave|151814.22222222222|
|Chiapas                        |151643.88888888888|
|Puebla                         |142642.44444444444|
|Ciudad de México               |139506.66666666666|
|Guanajuato                     |119046.33333333333|
|Michoacán de Ocampo            |102623.11111111111|
|Guerrero                       |93849.22222222222 |
|Nuevo León                     |93149.22222222222 |
+-------------------------------+------------------+
only showing top 10 rows


In [0]:
# Maximo y minimo de nacimientos por año
from pyspark.sql.functions import max, min

df.groupBy("year") \
  .agg(
      max("Total").alias("Maximo"),
      min("Total").alias("Minimo")
  ) \
  .show()

+----+------+------+
|year|Maximo|Minimo|
+----+------+------+
|2011|327165|  8921|
|2014|316088|  9096|
|2015|303778| 10776|
|2010|335898|  7362|
|2016|295635|  7856|
|2017|286204| 10842|
|2012|326412|  8911|
|2013|317834|  9303|
|2018|271684| 11375|
+----+------+------+



In [0]:
%sql
-- Crear vista para natalidad por estado en orden descendente
CREATE OR REPLACE VIEW vista_natalidad_estado AS

SELECT
Estado,
SUM(Total) AS Total_Nacimientos,
SUM(Hombres) AS Hombres,
SUM(Mujeres) AS Mujeres
FROM natalidad_flat
GROUP BY Estado;

--Consulta:

SELECT *
FROM vista_natalidad_estado
ORDER BY Total_Nacimientos DESC;

Estado,Total_Nacimientos,Hombres,Mujeres
México,2780698,1400196,1380391
Jalisco,1403088,713586,689501
Veracruz de Ignacio de la Llave,1366328,690867,675457
Chiapas,1364795,689436,675052
Puebla,1283782,644093,639465
Ciudad de México,1255560,630155,625403
Guanajuato,1071417,544133,527133
Michoacán de Ocampo,923608,467347,456258
Guerrero,844643,426726,417903
Nuevo León,838343,425442,412900


In [0]:
%sql
-- Crear vista para natalidad por año
CREATE OR REPLACE VIEW vista_natalidad_anual AS 
SELECT 
Year, 
SUM(Total) AS Total_Nacimientos
FROM natalidad_flat 
GROUP BY Year;

--Consulta:

SELECT *
FROM vista_natalidad_anual
ORDER BY Total_Nacimientos DESC;

Year,Total_Nacimientos
2010,2643908
2011,2586287
2012,2498880
2013,2478889
2014,2463420
2015,2353596
2016,2293708
2017,2234039
2018,2162535


In [0]:
%sql
--INNER JOIN entre las tablas creadas
SELECT
 g.Estado, 
 t.Year, 
 d.Total, 
 d.Hombres, 
 d.Mujeres 
 FROM natalidad_temporal t INNER JOIN natalidad_geografica g 
 ON t.nacimiento_id = g.nacimiento_id 
 
 INNER JOIN natalidad_demografica d 
 ON t.nacimiento_id = d.nacimiento_id;

Estado,Year,Total,Hombres,Mujeres
Aguascalientes,2010,26583,13603,12980
Baja California,2010,63559,32264,31295
Baja California Sur,2010,13988,7076,6912
Campeche,2010,20380,10230,10150
Coahuila de Zaragoza,2010,56972,28879,28093
Colima,2010,13796,7105,6691
Chiapas,2010,175382,87534,87789
Chihuahua,2010,74063,37236,36827
Ciudad de México,2010,160057,79504,80551
Durango,2010,42514,21290,21224


In [0]:
# INNER JOIN de las tablas temporales en Pyspark
df_join = df_temporal \
    .join(
        df_geografica,
        "nacimiento_id",
        "inner"
    ) \
    .join(
        df_demografica,
        "nacimiento_id",
        "inner"
    )

display(df_join)

nacimiento_id,Year,Estado,Total,Hombres,Mujeres,No_especificado
0,2010,Aguascalientes,26583,13603,12980,0
1,2010,Baja California,63559,32264,31295,0
2,2010,Baja California Sur,13988,7076,6912,0
3,2010,Campeche,20380,10230,10150,0
4,2010,Coahuila de Zaragoza,56972,28879,28093,0
5,2010,Colima,13796,7105,6691,0
6,2010,Chiapas,175382,87534,87789,59
7,2010,Chihuahua,74063,37236,36827,0
8,2010,Ciudad de México,160057,79504,80551,2
9,2010,Durango,42514,21290,21224,0


In [0]:
%sql
-- INNER JOIN para mostrar el total de nacimientos por estado y el promedio
SELECT g.Estado, 
SUM(d.Total) AS Total_Nacimientos,
AVG(d.Total) AS Promedio
 FROM natalidad_temporal t 
 
 INNER JOIN natalidad_geografica g 
ON t.nacimiento_id = g.nacimiento_id 
INNER JOIN natalidad_demografica d 
ON t.nacimiento_id = d.nacimiento_id 
GROUP BY g.Estado
 ORDER BY Total_Nacimientos DESC;

Estado,Total_Nacimientos,Promedio
México,2780698,308966.44444444444
Jalisco,1403088,155898.66666666666
Veracruz de Ignacio de la Llave,1366328,151814.22222222222
Chiapas,1364795,151643.88888888888
Puebla,1283782,142642.44444444444
Ciudad de México,1255560,139506.66666666666
Guanajuato,1071417,119046.33333333333
Michoacán de Ocampo,923608,102623.11111111111
Guerrero,844643,93849.22222222222
Nuevo León,838343,93149.22222222222
